In [ ]:
'''
python version 3.10.12
'''

In [ ]:
'''
Make sure to confirm the full path to the requirements.txt file. 
'''
! pip install -r requirements.txt
'''
Please restart this file after executing this cell !!!
'''

In [2]:
import argparse
import sys
import os
import time
import numpy as np
import torch
from torch import nn
from torch import Tensor
from torch.utils.data import DataLoader
import yaml
from tqdm import tqdm
from data_utils_rawboost import genSpoof_list,Dataset_ASVspoof2019_train,Dataset_ASVspoof2021_eval
from model import RawNet
# from tensorboardX import SummaryWriter
from core_scripts.startup_config import set_random_seed


In [ ]:



def evaluate_accuracy(dev_loader, model, device):
    num_correct = 0.0
    num_total = 0.0
    model.eval()
    for batch_x, batch_y in dev_loader:
        
        batch_size = batch_x.size(0)
        num_total += batch_size
        batch_x = batch_x.to(device)
        batch_y = batch_y.view(-1).type(torch.int64).to(device)
        batch_out = model(batch_x)
        _, batch_pred = batch_out.max(dim=1)
        num_correct += (batch_pred == batch_y).sum(dim=0).item()
    return 100 * (num_correct / num_total)


def produce_evaluation_file(dataset, model, device, save_path):
    data_loader = DataLoader(dataset, batch_size=24, shuffle=False, drop_last=False)
    model.eval()
    start_time=time.time()
    fname_list = []
    score_list = []  
    variant_list=[]
    random_list=[]
    src_list=[]
    label_list=[]
    for batch_x,utt_id,variant,random,source,label in tqdm(data_loader):
        
        batch_size = batch_x.size(0)
        batch_x = batch_x.to(device)
        batch_out = model(batch_x,is_test=True)
        batch_score = (batch_out[:, 1]
                       ).data.cpu().numpy().ravel()
        # add outputs
        variant_list.extend(variant)
        random_list.extend(random)
        src_list.extend(source)
        label_list.extend(label)
        fname_list.extend(utt_id)
        score_list.extend(batch_score.tolist())
    end_time=time.time()
    use_time=end_time-start_time
    print(f'Use_time: {use_time}s')    
    with open(save_path, 'w') as fh:
        for v,f,r,s, cm,l in zip(variant_list,fname_list,random_list,src_list,score_list,label_list):
            fh.write('{} {} {} {} {} {}\n'.format(v,f,r,s, cm,l))
    fh.close()   
    print('Scores saved to {}'.format(save_path))

def train_epoch(train_loader, model, lr,optim, device):
    running_loss = 0
    num_correct = 0.0
    num_total = 0.0
    ii = 0
    model.train()

    #set objective (Loss) functions
    weight = torch.FloatTensor([0.1, 0.9]).to(device)
    criterion = nn.CrossEntropyLoss(weight=weight)
    
    for batch_x, batch_y in train_loader:
       
        batch_size = batch_x.size(0)
        num_total += batch_size
        ii += 1
        batch_x = batch_x.to(device)
        batch_y = batch_y.view(-1).type(torch.int64).to(device)
        batch_out = model(batch_x)
        batch_loss = criterion(batch_out, batch_y)
        _, batch_pred = batch_out.max(dim=1)
        num_correct += (batch_pred == batch_y).sum(dim=0).item()
        running_loss += (batch_loss.item() * batch_size)
        if ii % 10 == 0:
            sys.stdout.write('\r \t {:.2f}'.format(
                (num_correct/num_total)*100))
        optim.zero_grad()
        batch_loss.backward()
        optim.step()
       
    running_loss /= num_total
    train_accuracy = (num_correct/num_total)*100
    return running_loss, train_accuracy
def eval_test(output_path,eval_path):
    dir_yaml='./model_config_RawNet.yaml'

    with open(dir_yaml, 'r') as f_yaml:
            parser1 = yaml.load(f_yaml,Loader=yaml.FullLoader)

    if not os.path.exists('models'):
        os.mkdir('models')
    # args = parser.parse_args()
   
    #make experiment reproducible
    set_random_seed(1234)
    
    track = 'LA'

    assert track in ['LA', 'PA','DF'], 'Invalid track given'

    device = 'cuda'            
    print('Device: {}'.format(device))
    
    #model 
    model = RawNet(parser1['model'], device)
    nb_params = sum([param.view(-1).size()[0] for param in model.parameters()])
    model =(model).to(device)
    
   
    optimizer = torch.optim.Adam(model.parameters(), lr=0.0001,weight_decay=0.0001)
  
    model_path='change this to your RawBoost model' # download here [https://huggingface.co/VoiceWukong/VoiceWukong/resolve/main/RawBoost.pth?download=true]
  
    model.load_state_dict(torch.load(model_path,map_location=device))
    print('Model loaded : {}'.format(model_path))
    variant_list,file_list,random_list,src_list,label_list = genSpoof_list( dir_meta = eval_path ,is_train=False,is_eval=True)
    print('no. of eval trials',len(file_list))
    '''
    |- path to VoiceWukong dataset
        |- Alldataset
        |- Alldataset32K
        |- ...
    '''
    eval_set=Dataset_ASVspoof2021_eval(variant_IDs=variant_list,list_IDs = file_list,random_IDs=random_list,src_IDs=src_list,fake_IDs=label_list,base_dir = 'change this to your VoiceWukong dataset path')
    produce_evaluation_file(eval_set, model, device, output_path)
    return 
    
eval_test('change this to the path to save zh_eval_score.txt','change this to your zh_eval_list.txt')
eval_test('change this to the path to save en_eval_score.txt','change this to your eval_list.txt')

